# Testing Methodology — Writing Tests for Robotics Code

**Reference:** pytest documentation — https://docs.pytest.org/en/stable/getting-started.html  
**Reference:** "The Art of Unit Testing" — Roy Osherove  

---

## Why Write Tests?

Imagine you write a function today, it works, and six weeks later you change something unrelated and it silently breaks. Without tests, you might not notice until the robot arm moves somewhere unexpected.

Tests are a safety net. They let you:
- Confirm a function does what you think it does
- Catch breakages immediately when you change code elsewhere
- Prove to yourself (and others) that the physics is correct

The core habit: **run tests every time you touch the code.**

---

## How pytest Discovers Tests

pytest is opinionated about naming. It automatically finds tests by looking for:

| Rule | Example |
|---|---|
| Files named `test_*.py` or `*_test.py` | `test_kinematics.py` ✓ |
| Functions starting with `test_` | `def test_eye_transform():` ✓ |
| Functions starting with `test` (no underscore) | `def testCheck():` — **also discovered**, careful! |

This is why your `testCheck` helper caused an error — pytest tried to run it as a test. Naming helper functions `assert_close`, `check_matrix`, etc. (no `test` prefix) keeps them invisible to pytest.

---

## Exact Equality vs. np.allclose

You cannot use `==` to compare floating-point numbers reliably. This is a fundamental property of how computers store decimals:

```python
0.1 + 0.2 == 0.3  # False — floating point rounding
```

Instead, compare with a tolerance:

```python
np.allclose(result, expected)         # tolerances: rtol=1e-5, atol=1e-8
np.allclose(result, expected, atol=1e-6)  # tighter absolute tolerance
```

`np.allclose` returns `True` if every element satisfies:
$$|a - b| \leq \text{atol} + \text{rtol} \times |b|$$

For robotics: the default tolerances are usually fine. Only tighten them if you have a reason.

---

## Testing Physical Properties vs. Exact Numbers

There are two kinds of tests:

### 1. Known-answer tests
You know exactly what the output should be, so you hard-code it.

> *"A joint with no rotation and no translation should give the identity matrix."*

These are easy to write for simple cases but hard to scale — you cannot always compute the expected answer by hand.

### 2. Property-based tests
You don't know the exact numbers, but you know a mathematical property the result must always satisfy.

> *"The mass matrix must always be symmetric."*  
> *"All eigenvalues of the mass matrix must be positive."*

These are more powerful — they catch bugs in configurations you never explicitly tested.

For this robotics project, **property-based tests are your most valuable tool** for the dynamics code.

---

## The Arrange–Act–Assert Pattern

Every well-structured test has three parts:

```python
def test_mass_matrix_positive_definite():
    # Arrange — set up the inputs
    joints = [{'mass': 1.0, 'center_of_mass': [0.5, 0, 0], ...}]
    theta_syms = ...

    # Act — call the function under test
    M = build_mass_matrix(T_cum, joints, theta_syms)
    M_num = evaluate_M_at_zero(M, theta_syms)

    # Assert — check the result
    eigenvalues = np.linalg.eigvals(M_num)
    assert np.all(eigenvalues > 0)
```

Keeping these three parts visually separated makes tests easy to read and debug.

---

## Test-Driven Development (TDD)

TDD flips the usual order:

1. Write a test that describes the behaviour you want
2. Run it — it **fails** (the function doesn't exist yet)
3. Write the minimum code to make it pass
4. Refactor if needed, tests stay green

The failing test first is intentional — it proves the test actually detects a broken state. A test that passes before the code exists is not testing anything.

You don't have to use TDD strictly, but it is worth trying on at least one function to experience how it shapes your thinking.

---

## What to Test in This Project

### Kinematics (`test_kinematics.py`) — already done

| Test | What it checks |
|---|---|
| Identity transform | All-zero DH params → 4×4 identity |
| Pure translation | `a=1` → shifts x by 1, nothing else changes |
| Pure rotation | `theta=π/2` → correct rotation entries |
| Two-joint chain | Two `a=1` joints → end-effector at x=2 |

### Dynamics (`test_dynamics.py`) — next

| Test | What it checks | Why |
|---|---|---|
| Mass matrix symmetry | `M == M.T` | Required by physics |
| Mass matrix positive definite | All eigenvalues > 0 | Energy must always be positive |
| Gravity vector at zero config | Known-answer for simple case | Validates differentiation of V |
| State derivative at rest, no torque | `theta_ddot == 0` | Zero torque + zero velocity → no acceleration |

---

## Running Your Tests

```bash
# Run all tests
.venv/bin/pytest

# Run one file, verbose
.venv/bin/pytest tests/test_dynamics.py -v

# Stop at first failure
.venv/bin/pytest -x
```

Green = nothing broke. Red = something broke, read the traceback, fix before moving on.